In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import load_model

base_dados = '../data/iniciacao.csv'
df = pd.read_csv(base_dados)

#pegar os valores direto
perguntas = df['perguntas'].values
respostas = df['respostas'].values

#todo o processo de desenvolvimento da RNN será uma LSTM com o Keras API
tokenizer = Tokenizer(num_words=5000) #limite de palavras
tokenizer.fit_on_texts(perguntas)
perguntas_token = tokenizer.texts_to_sequences(perguntas) #transforma todas as palavras em sequencias numericas (coluna x)
x = pad_sequences(perguntas_token, maxlen=20, padding='post')

#preciso aprender depois sobre Categorical Encoding
tokenizer = Tokenizer(num_words=5000) #limite de palavras
tokenizer.fit_on_texts(respostas)
respostas_token = tokenizer.texts_to_sequences(respostas) #transforma todas as palavras em sequencias numericas (coluna y)
y = pad_sequences(respostas_token, maxlen=20, padding='post')

        
vocab = len(tokenizer.word_index) + 1 #numero de palavras unicas no dicionario
saida = len(df['respostas'].unique()) #numero de respostas que o bot ira fornecer

model = Sequential()

#Embedding transforma os IDs em vetores densos e é aqui que ela ira aprender o "significado" e contexto das palavras
model.add(Embedding(input_dim=vocab, output_dim=64, input_length=20))

#LSTM será a memoria para processar a sequência de vetores das palavras e de certa forma lembrar o contexto
model.add(LSTM(64))

#É a saida de decisão, pega os valores de LSTM e decide a resposta a retornar
model.add(Dense(saida, activation='softmax'))
model.summary() #mostrar arquitetura

#verificar sobre o y_train

#Compilar o modelo
model.compile(loss='categorical_crossentropy', # Função de perda para classificação
              optimizer='adam',                 # Otimizador padrão
              metrics=['accuracy'])             # Queremos ver a acurácia


#Treinamento é aqui
model.fit(x, y, epochs=50, batch_size=32)

#Salvar o modelo que treinei
model.save('../models/chatbotIA.h5')
        

c:\Users\playe\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 20), output.shape=(None, 4)